In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load


# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
import pandas as pd
import cv2
import string

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# 1. Frame the problem
Using the customer description, Define the problem your trying to solve in your own words (remember this is not technial but must be specific so the customer understands the project

We want to build an automated Tetris player than will play Tetris in an headless mode environment (which allows the bot to play without waiting for regular game delay). The goal is to maximize how long the bot lasts before losing. To solve the problem, we will use a machine learning model adapted to the task to train our bot player.

# 2. Get the Data 
Define how you recieved the data (provided, gathered..)

We were given a link to an implementation of tetris: https://gitlab.com/yukiman/tetris_a. This repository also allows the game to run in headless mode, and provides several bot implementations.

# 3. Explore the Data
Gain insights into the data you have from step 2, making sure to identify any bias

In [45]:
import subprocess

def run_strategy(strategy):
    os.system("clear")
    result = subprocess.run(
        ["python", "tetris_a/src/main.py", strategy, "--nodisplay"],
        capture_output=True,
        text=True
    )
    all_output = result.stderr + result.stdout
    return all_output.count('\n') - 1

trials = 10
mcts_average = sum([run_strategy("mcts") for i in range(trials)]) / trials
random_average = sum([run_strategy("randomChoice") for i in range(trials)]) / trials
genetic_average = sum([run_strategy("genetic") for i in range(trials)]) / trials
greedy_average = sum([run_strategy("greedy") for i in range(trials)]) / trials
print(f"Monte Carlo: {mcts_average}")
print(f"Random: {random_average}")
print(f"Genetic: {genetic_average}")
print(f"Greedy: {greedy_average}")

Monte Carlo: 12.8
Random: 3.0
Genetic: 17.4
Greedy: 8331.9


We used the existing models from the GitHub repository to get a sense of how strong our bot should be. The genetic model performs very poorly, and so it has likely not been trained yet. The Monte Carlo Tree Search also performs poorly, however, and I do not know why this is the case. The greedy algorithm is the only viable bot player, and over 10 simulations it survived for an average of 8331.9 moves. We will take the greedy algorithm as a benchmark score for the model we will train. However, we should expect that our genetic model will perform much better than the simple greedy algorithm since the genetic algorithm will actually "learn".

# 4.Prepare the Data


Apply any data transformations and explain what and why


Our genetic model will learn as it plays. The game itself is the data, and so no data transformations are necessary. Here we will outline how our bot will train:

We will select certain genes for our model, and each bot will have assigned weights for each gene, which corresponds to some feature of the game board. The bot will then pick the move that minimizes the weighted sum, and we will use an elitist survival model, along with reproduction and mutation to create new bots in the next generation. The genes we will use are number of lines cleared, sum of column heights, maximum column height, number of empty cells with a filled square above, sum of absolute height differences between columns for now (lines, total height, bumpiness, and holes for now).

# 5. Model the data
Using selected ML models, experment with your choices and describe your findings. Finish by selecting a Model to continue with


LogisticRegression(max_iter=1000, random_state=1434)


Random Forest Model Accuracy: 75.336323%

Logistic Model Accuracy: 78.026906%


# 6. Fine Tune the Model

With the select model descibe the steps taken to acheve the best rusults possiable 



Model Accuracy: 78.026906%


# 7. Present
In a customer faceing Document provide summery of finding and detail approach taken


# 8. Launch the Model System
Define your production run code, This should be self susficent and require only your model pramaters 
